# Subreddit Recommendation: NCF Model Training

This notebook trains **3 NCF models** for subreddit recommendation:

| Model | Data | Text Embeddings | Description |
|-------|------|-----------------|-------------|
| **Model A** | v3.5 (tfidf) | 128-dim (filtered) | NCF + Qwen3 Text Embeddings |
| **Model B** | v4 | None | NCF Baseline (no text) |
| **Model C** | v4 | 128-dim (filtered) | NCF + Qwen3 Text Embeddings |
| **Model D** | v3.5 (tfidf) | 128-dim (filtered) | NCF + Qwen3 Text Embeddings + Gated Fusion|
| **Model E** | v4 | 128-dim (filtered) | NCF + Qwen3 Text Embeddings + Gated Fusion|
| **Model F** | v4 (chi-sq) | 128-dim (filtered) | NCF + Qwen3 Text Embeddings|
| **Model G** | v4 (chi-sq) | 128-dim (filtered) | NCF + Qwen3 Text Embeddings + Gated Fusion|

**Model B** serves as the baseline to compare the effect of text embeddings.

## Expected Directory Structure

```
CIS 5300/
├── engage_corpus_processed_v3_5tfidf/
│   ├── ncf_data/
│   │   ├── train.tsv
│   │   ├── dev.tsv
│   │   └── test.tsv
│   ├── user_mapping.json
│   └── subreddit_mapping.json
├── engage_corpus_processed_v3_5_chisq/
│   ├── ncf_data/
│   │   ├── train.tsv
│   │   ├── dev.tsv
│   │   └── test.tsv
│   ├── user_mapping.json
│   └── subreddit_mapping.json
├── engage_corpus_processed_v4/
│   ├── ncf_data/
│   │   ├── train.tsv
│   │   ├── dev.tsv
│   │   └── test.tsv
│   ├── user_mapping.json
│   └── subreddit_mapping.json
├── engage_corpus_processed_v4_chi2/
│   ├── ncf_data/
│   │   ├── train.tsv
│   │   ├── dev.tsv
│   │   └── test.tsv
│   ├── user_mapping.json
│   └── subreddit_mapping.json
└── embeddings_qwen3_8b/
    ├── v3_5tfidf/
    │   └── filtered/
    │       ├── train/
    │       │   ├── user_embeddings_128.npy
    │       │   └── subreddit_embeddings_128.npy
    │       └── dev/
    │           ├── user_embeddings_128.npy
    │           └── subreddit_embeddings_128.npy
    ├── v3_5_chisq/
    │   └── filtered/
    │       ├── train/
    │       │   ├── user_embeddings_128.npy
    │       │   └── subreddit_embeddings_128.npy
    │       └── dev/
    │           ├── user_embeddings_128.npy
    │           └── subreddit_embeddings_128.npy
    ├── v4/
    │   └── filtered/
    │       ├── train/
    │       │   ├── user_embeddings_128.npy
    │       │   └── subreddit_embeddings_128.npy
    │       └── dev/
    │           ├── user_embeddings_128.npy
    │           └── subreddit_embeddings_128.npy
    └── v4_chi2/
        └── filtered/
            ├── train/
            │   ├── user_embeddings_128.npy
            │   └── subreddit_embeddings_128.npy
            └── dev/
                ├── user_embeddings_128.npy
                └── subreddit_embeddings_128.npy
```

## 1. Setup

In [ ]:
# Install required packages
!pip install -q torch numpy tqdm scikit-learn

In [ ]:
# Mount Google Drive (for Colab)
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
import os
from tqdm import tqdm
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

## 2. Configuration

In [ ]:
# ============================================================
# PATHS - Update these to match your Google Drive structure
# ============================================================

BASE_DIR = '/content/drive/MyDrive/CIS 5300'

# Data directories
DATA_DIRS = {
    'v3_5tfidf': os.path.join(BASE_DIR, 'engage_corpus_processed_v3_5tfidf'),
    'v4': os.path.join(BASE_DIR, 'engage_corpus_processed_v4'),
    # --- Chi-Sq Data ---
    'v3_5_chi2': os.path.join(BASE_DIR, 'engage_corpus_processed_v3_5_chi2'),
    'v4_chi2':   os.path.join(BASE_DIR, 'engage_corpus_processed_v4_chi2'),
}

# Embedding directories
EMBEDDING_DIRS = {
    'v3_5tfidf_filtered': os.path.join(BASE_DIR, 'embeddings_qwen3_8b', 'v3_5tfidf', 'filtered'),
    'v4_filtered': os.path.join(BASE_DIR, 'embeddings_qwen3_8b', 'v4', 'filtered'),
    # --- Chi-Sq Embed ---
    'v3_5_chi2_filtered': os.path.join(BASE_DIR, 'embeddings_qwen3_8b', 'v3_5_chi2', 'filtered'),
    'v4_chi2_filtered':   os.path.join(BASE_DIR, 'embeddings_qwen3_8b', 'v4_chi2', 'filtered'),
}

# Output directories
OUTPUT_DIR = os.path.join(BASE_DIR, 'trained_models')
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ============================================================
# HYPERPARAMETERS
# ============================================================

EMBEDDING_DIM = 64       # NCF embedding dimension
TEXT_EMB_DIM = 128       # Qwen3 MRL embedding dimension
TEXT_PROJ_DIM = 128      # Projected text embedding dimension
BATCH_SIZE = 2096
LEARNING_RATE = 0.0001
NUM_EPOCHS = 90
CHECKPOINT_EVERY = 3     # Save checkpoint every N epochs
NUM_NEGATIVES = 4        # Negative samples per positive

print(f"Output directory: {OUTPUT_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

In [ ]:
# Verify directories exist
print("Verifying directories...\n")

all_ok = True

for name, path in DATA_DIRS.items():
    ncf_path = os.path.join(path, 'ncf_data')
    if os.path.exists(ncf_path):
        files = os.listdir(ncf_path)
        print(f"✓ {name} NCF data: {files}")
    else:
        print(f"✗ {name} NCF data NOT FOUND: {ncf_path}")
        all_ok = False

print()

for name, path in EMBEDDING_DIRS.items():
    train_path = os.path.join(path, 'train')
    if os.path.exists(train_path):
        files = os.listdir(train_path)
        print(f"✓ {name} embeddings: {files}")
    else:
        print(f"✗ {name} embeddings NOT FOUND: {train_path}")
        all_ok = False

if all_ok:
    print("\n✓ All directories found!")
else:
    print("\n⚠ Some directories missing - check paths above")

## 3. Model Definitions

In [ ]:
def identify_nan_users(embeddings, user_ids_list):
    """Identify which users have NaN in their embeddings."""
    has_nan = np.isnan(embeddings).any(axis=1)
    nan_indices = np.where(has_nan)[0]
    nan_user_ids = {user_ids_list[idx] for idx in nan_indices}
    print(f"  Found {len(nan_user_ids)} users with NaN embeddings")
    return nan_user_ids

def clean_embeddings_inplace(embeddings):
    """Clean embeddings by replacing NaN values with zeros."""
    nan_mask = np.isnan(embeddings)
    num_nans = nan_mask.sum()
    if num_nans > 0:
        print(f"  Replacing {num_nans} NaN values with zeros")
        embeddings[nan_mask] = 0.0
    return embeddings

In [ ]:
class NCFBaseline(nn.Module):
    """
    NCF Baseline Model (No Text Embeddings)

    Uses only collaborative filtering signals:
    - GMF (Generalized Matrix Factorization) branch
    - MLP (Multi-Layer Perceptron) branch
    - NeuMF fusion layer
    """
    def __init__(self, num_users, num_subreddits, embedding_dim=64):
        super().__init__()

        # GMF embeddings
        self.user_embedding_gmf = nn.Embedding(num_users, embedding_dim)
        self.sub_embedding_gmf = nn.Embedding(num_subreddits, embedding_dim)

        # MLP embeddings
        self.user_embedding_mlp = nn.Embedding(num_users, embedding_dim)
        self.sub_embedding_mlp = nn.Embedding(num_subreddits, embedding_dim)

        # MLP layers
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64)
        )

        # Final prediction layer
        self.final = nn.Linear(embedding_dim + 64, 1)

        # Initialize embeddings
        nn.init.normal_(self.user_embedding_gmf.weight, std=0.01)
        nn.init.normal_(self.sub_embedding_gmf.weight, std=0.01)
        nn.init.normal_(self.user_embedding_mlp.weight, std=0.01)
        nn.init.normal_(self.sub_embedding_mlp.weight, std=0.01)

    def forward(self, user, subreddit):
        # GMF branch
        user_emb_gmf = self.user_embedding_gmf(user)
        sub_emb_gmf = self.sub_embedding_gmf(subreddit)
        gmf_output = user_emb_gmf * sub_emb_gmf  # Element-wise product

        # MLP branch
        user_emb_mlp = self.user_embedding_mlp(user)
        sub_emb_mlp = self.sub_embedding_mlp(subreddit)
        mlp_input = torch.cat([user_emb_mlp, sub_emb_mlp], dim=-1)
        mlp_output = self.mlp(mlp_input)

        # NeuMF fusion
        fusion = torch.cat([gmf_output, mlp_output], dim=-1)
        return self.final(fusion).squeeze()

In [ ]:
class NCFWithTextEmbeddings(nn.Module):
    """
    NCF Model with Text Embeddings

    Enhances NCF with Qwen3 text embeddings:
    - Projects text embeddings to match NCF dimension
    - Concatenates projected text with NCF embeddings
    - Uses enhanced representations in both GMF and MLP branches
    """
    def __init__(self, num_users, num_subreddits, embedding_dim=64,
                 text_emb_dim=128, text_proj_dim=128):
        super().__init__()

        # Text embedding projection layers
        self.user_text_proj = nn.Linear(text_emb_dim, text_proj_dim)
        self.sub_text_proj = nn.Linear(text_emb_dim, text_proj_dim)

        # Enhanced dimension = NCF embedding + projected text
        enhanced_dim = embedding_dim + text_proj_dim

        # GMF embeddings
        self.user_embedding_gmf = nn.Embedding(num_users, embedding_dim)
        self.sub_embedding_gmf = nn.Embedding(num_subreddits, embedding_dim)

        # MLP embeddings
        self.user_embedding_mlp = nn.Embedding(num_users, embedding_dim)
        self.sub_embedding_mlp = nn.Embedding(num_subreddits, embedding_dim)

        # MLP layers (input is 2x enhanced_dim)
        self.mlp = nn.Sequential(
            nn.Linear(enhanced_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64)
        )

        # Final prediction layer
        self.final = nn.Linear(enhanced_dim + 64, 1)

        # Initialize embeddings
        nn.init.normal_(self.user_embedding_gmf.weight, std=0.01)
        nn.init.normal_(self.sub_embedding_gmf.weight, std=0.01)
        nn.init.normal_(self.user_embedding_mlp.weight, std=0.01)
        nn.init.normal_(self.sub_embedding_mlp.weight, std=0.01)

    def forward(self, user, subreddit, user_text_emb, sub_text_emb):
        # Project text embeddings
        user_text_proj = self.user_text_proj(user_text_emb)
        sub_text_proj = self.sub_text_proj(sub_text_emb)

        # GMF branch with text enhancement
        user_emb_gmf = self.user_embedding_gmf(user)
        sub_emb_gmf = self.sub_embedding_gmf(subreddit)
        enhanced_user_gmf = torch.cat([user_emb_gmf, user_text_proj], dim=-1)
        enhanced_sub_gmf = torch.cat([sub_emb_gmf, sub_text_proj], dim=-1)
        gmf_output = enhanced_user_gmf * enhanced_sub_gmf

        # MLP branch with text enhancement
        user_emb_mlp = self.user_embedding_mlp(user)
        sub_emb_mlp = self.sub_embedding_mlp(subreddit)
        enhanced_user_mlp = torch.cat([user_emb_mlp, user_text_proj], dim=-1)
        enhanced_sub_mlp = torch.cat([sub_emb_mlp, sub_text_proj], dim=-1)
        mlp_input = torch.cat([enhanced_user_mlp, enhanced_sub_mlp], dim=-1)
        mlp_output = self.mlp(mlp_input)

        # NeuMF fusion
        fusion = torch.cat([gmf_output, mlp_output], dim=-1)
        return self.final(fusion).squeeze()

In [ ]:
class GatedFusion(nn.Module):
    """
    Fuses an ID embedding with a Text embedding using a learnable gate.
    """
    def __init__(self, id_dim, text_dim, dropout=0.2):
        super(GatedFusion, self).__init__()
        self.text_proj = nn.Linear(text_dim, id_dim)
        self.dropout = nn.Dropout(dropout)
        self.gate_net = nn.Linear(id_dim * 2, id_dim)

    def forward(self, id_emb, text_emb):
        text_feat = F.relu(self.text_proj(text_emb))
        text_feat = self.dropout(text_feat)
        # Gate calculation
        combined = torch.cat([id_emb, text_feat], dim=1)
        gate = torch.sigmoid(self.gate_net(combined))
        # Residual fusion
        return id_emb + (gate * text_feat)

class NCFWithTextGated(nn.Module):
    """
    NCF Model that uses GatedFusion instead of simple concatenation.
    """
    def __init__(self, num_users, num_subreddits, embedding_dim=64,
                 text_emb_dim=128, text_proj_dim=128):
        super().__init__()

        # --- Gated Fusion Modules ---
        self.user_fusion = GatedFusion(embedding_dim, text_emb_dim)
        self.sub_fusion = GatedFusion(embedding_dim, text_emb_dim)

        # --- Base Embeddings ---
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.sub_embedding = nn.Embedding(num_subreddits, embedding_dim)

        # --- MLP Branch (Note: Input dim is smaller than concat model) ---
        # Because we fuse before the MLP, the input is just [User_Fused, Item_Fused]
        # Size = 64 + 64 = 128
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64)
        )

        self.final = nn.Linear(embedding_dim + 64, 1)

        # Init weights
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.sub_embedding.weight)
        # Initialize gate bias to -1 to start by trusting ID more (stability)
        nn.init.constant_(self.user_fusion.gate_net.bias, -1.0)
        nn.init.constant_(self.sub_fusion.gate_net.bias, -1.0)

    def forward(self, user, subreddit, user_text_emb, sub_text_emb):
        u_id = self.user_embedding(user)
        s_id = self.sub_embedding(subreddit)

        # Apply Gating
        u_rich = self.user_fusion(u_id, user_text_emb)
        s_rich = self.sub_fusion(s_id, sub_text_emb)

        # Standard NCF logic using the rich vectors
        gmf_out = u_rich * s_rich
        mlp_in = torch.cat([u_rich, s_rich], dim=1)
        mlp_out = self.mlp(mlp_in)

        fusion = torch.cat([gmf_out, mlp_out], dim=-1)
        return self.final(fusion).squeeze()

## 4. Dataset Classes

In [ ]:
class NCFDatasetBaseline(Dataset):
    """
    Dataset for NCF baseline (no text embeddings).

    Loads interactions from TSV file and generates negative samples.
    """
    def __init__(self, interactions_file, num_subreddits, num_negatives=4):
        self.num_negatives = num_negatives
        self.num_subreddits = num_subreddits

        # Load interactions
        self.interactions = []
        with open(interactions_file, 'r') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    self.interactions.append((int(parts[0]), int(parts[1])))

        # Build user-item set for negative sampling
        self.user_items = {}
        for user_id, sub_id in self.interactions:
            if user_id not in self.user_items:
                self.user_items[user_id] = set()
            self.user_items[user_id].add(sub_id)

        print(f"  Loaded {len(self.interactions)} interactions from {os.path.basename(interactions_file)}")

    def __len__(self):
        return len(self.interactions) * (1 + self.num_negatives)

    def __getitem__(self, idx):
        interaction_idx = idx // (1 + self.num_negatives)
        sample_idx = idx % (1 + self.num_negatives)
        user_id, pos_sub_id = self.interactions[interaction_idx]

        if sample_idx == 0:
            # Positive sample
            return (torch.tensor(user_id),
                    torch.tensor(pos_sub_id),
                    torch.tensor(1.0))
        else:
            # Negative sample
            neg_sub_id = np.random.randint(0, self.num_subreddits)
            while neg_sub_id in self.user_items.get(user_id, set()):
                neg_sub_id = np.random.randint(0, self.num_subreddits)
            return (torch.tensor(user_id),
                    torch.tensor(neg_sub_id),
                    torch.tensor(0.0))

In [ ]:
class NCFDatasetWithText(Dataset):
    """
    Dataset for NCF with text embeddings.

    Uses mapping files to correctly index embeddings, since not all users/subreddits
    may have embeddings in every split.
    """
    def __init__(self, interactions_file, user_embeddings, sub_embeddings,
                 num_subreddits, num_negatives=4,
                 user_ids_file=None, subreddit_names_file=None, subreddit_mapping=None):
        """
        Args:
            interactions_file: Path to train/dev/test.tsv
            user_embeddings: numpy array of user embeddings
            sub_embeddings: numpy array of subreddit embeddings
            num_subreddits: Total number of subreddits
            num_negatives: Number of negative samples per positive
            user_ids_file: Path to user_ids.json from embedding script
            subreddit_names_file: Path to subreddit_names.json from embedding script
            subreddit_mapping: Dict with 'subreddit2idx' to map names to indices
        """
        self.num_negatives = num_negatives
        self.num_subreddits = num_subreddits

        # Build user_id -> embedding_index mapping
        if user_ids_file and os.path.exists(user_ids_file):
            with open(user_ids_file, 'r') as f:
                user_ids_list = json.load(f)
            # user_ids_list[i] = the user_id at embedding index i
            # We need: user_id -> embedding index
            self.user_id_to_emb_idx = {uid: idx for idx, uid in enumerate(user_ids_list)}
            print(f"  Loaded user ID mapping: {len(self.user_id_to_emb_idx)} users have embeddings")

            # Identify nan embeddings
            self.nan_user_ids = identify_nan_users(user_embeddings, user_ids_list)
        else:
            # Assume 1:1 mapping (embedding index = user_id)
            self.user_id_to_emb_idx = None
            self.nan_user_ids = set()

        # Build subreddit_id -> embedding_index mapping
        if subreddit_names_file and os.path.exists(subreddit_names_file) and subreddit_mapping:
            with open(subreddit_names_file, 'r') as f:
                sub_names_list = json.load(f)
            # sub_names_list[i] = subreddit name at embedding index i
            # subreddit_mapping['subreddit2idx'][name] = subreddit_id in NCF data
            # We need: subreddit_id -> embedding index
            subreddit2idx = subreddit_mapping['subreddit2idx']
            self.sub_id_to_emb_idx = {}
            for emb_idx, sub_name in enumerate(sub_names_list):
                if sub_name in subreddit2idx:
                    sub_id = subreddit2idx[sub_name]
                    self.sub_id_to_emb_idx[sub_id] = emb_idx
            print(f"  Loaded subreddit ID mapping: {len(self.sub_id_to_emb_idx)} subreddits have embeddings")
        else:
            # Assume 1:1 mapping
            self.sub_id_to_emb_idx = None

        # Load and filter interactions
        self.interactions = []
        skipped_user = 0
        skipped_sub = 0
        skipped_nan_user = 0

        with open(interactions_file, 'r') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    user_id = int(parts[0])
                    sub_id = int(parts[1])

                    # Check if user has NaN embedding - SKIP IF TRUE
                    if user_id in self.nan_user_ids:
                        skipped_nan_user += 1
                        continue

                    # Check if we have embeddings for both
                    has_user_emb = (self.user_id_to_emb_idx is None or
                                    user_id in self.user_id_to_emb_idx)
                    has_sub_emb = (self.sub_id_to_emb_idx is None or
                                   sub_id in self.sub_id_to_emb_idx)

                    if has_user_emb and has_sub_emb:
                        self.interactions.append((user_id, sub_id))
                    else:
                        if not has_user_emb:
                            skipped_user += 1
                        if not has_sub_emb:
                            skipped_sub += 1

        # Build user-item set for negative sampling
        self.user_items = {}
        for user_id, sub_id in self.interactions:
            if user_id not in self.user_items:
                self.user_items[user_id] = set()
            self.user_items[user_id].add(sub_id)

        # Valid subreddit IDs for negative sampling (those with embeddings)
        if self.sub_id_to_emb_idx is not None:
            self.valid_sub_ids = list(self.sub_id_to_emb_idx.keys())
        else:
            self.valid_sub_ids = list(range(num_subreddits))

        # Store embeddings
        self.user_embeddings = torch.FloatTensor(user_embeddings)
        self.sub_embeddings = torch.FloatTensor(sub_embeddings)

        print(f"  Loaded {len(self.interactions)} interactions from {os.path.basename(interactions_file)}")
        if skipped_user > 0 or skipped_sub > 0 or skipped_nan_user > 0:
            print(f"  ⚠ Skipped {skipped_user} (no user emb) + "
                  f"{skipped_sub} (no sub emb) + "
                  f"{skipped_nan_user} (NaN user emb) interactions")
        print(f"  User embeddings: {self.user_embeddings.shape}")
        print(f"  Subreddit embeddings: {self.sub_embeddings.shape}")
        print(f"  Valid subreddits for neg sampling: {len(self.valid_sub_ids)}")

    def __len__(self):
        return len(self.interactions) * (1 + self.num_negatives)

    def _get_user_emb(self, user_id):
        """Get embedding for a user_id using the mapping."""
        if self.user_id_to_emb_idx is not None:
            emb_idx = self.user_id_to_emb_idx[user_id]
        else:
            emb_idx = user_id
        return self.user_embeddings[emb_idx]

    def _get_sub_emb(self, sub_id):
        """Get embedding for a subreddit_id using the mapping."""
        if self.sub_id_to_emb_idx is not None:
            emb_idx = self.sub_id_to_emb_idx[sub_id]
        else:
            emb_idx = sub_id
        return self.sub_embeddings[emb_idx]

    def __getitem__(self, idx):
        interaction_idx = idx // (1 + self.num_negatives)
        sample_idx = idx % (1 + self.num_negatives)
        user_id, pos_sub_id = self.interactions[interaction_idx]

        if sample_idx == 0:
            # Positive sample
            return (torch.tensor(user_id),
                    torch.tensor(pos_sub_id),
                    torch.tensor(1.0),
                    self._get_user_emb(user_id),
                    self._get_sub_emb(pos_sub_id))
        else:
            # Negative sample - pick from valid subreddits only
            neg_sub_id = self.valid_sub_ids[np.random.randint(0, len(self.valid_sub_ids))]
            while neg_sub_id in self.user_items.get(user_id, set()):
                neg_sub_id = self.valid_sub_ids[np.random.randint(0, len(self.valid_sub_ids))]

            return (torch.tensor(user_id),
                    torch.tensor(neg_sub_id),
                    torch.tensor(0.0),
                    self._get_user_emb(user_id),
                    self._get_sub_emb(neg_sub_id))

## 5. Training Functions

In [ ]:
def train_epoch_baseline(model, dataloader, optimizer, criterion, device):
    """Train one epoch for baseline model."""
    model.train()
    total_loss = 0
    num_batches = 0

    for batch in tqdm(dataloader, desc="  Training", leave=False):
        user, subreddit, label = batch
        user = user.to(device)
        subreddit = subreddit.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        pred = model(user, subreddit)
        loss = criterion(pred, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches


def validate_baseline(model, dataloader, criterion, device):
    """Validate baseline model."""
    model.eval()
    total_loss = 0
    num_batches = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="  Validating", leave=False):
            user, subreddit, label = batch
            user = user.to(device)
            subreddit = subreddit.to(device)
            label = label.to(device)

            pred = model(user, subreddit)
            loss = criterion(pred, label)

            total_loss += loss.item()
            num_batches += 1

    return total_loss / num_batches

In [ ]:
def train_epoch_with_text(model, dataloader, optimizer, criterion, device):
    """Train one epoch for model with text embeddings."""
    model.train()
    total_loss = 0
    num_batches = 0

    for batch in tqdm(dataloader, desc="  Training", leave=False):
        user, subreddit, label, user_emb, sub_emb = batch
        user = user.to(device)
        subreddit = subreddit.to(device)
        label = label.to(device)
        user_emb = user_emb.to(device)
        sub_emb = sub_emb.to(device)

        optimizer.zero_grad()
        pred = model(user, subreddit, user_emb, sub_emb)
        loss = criterion(pred, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches


def validate_with_text(model, dataloader, criterion, device):
    """Validate model with text embeddings."""
    model.eval()
    total_loss = 0
    num_batches = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="  Validating", leave=False):
            user, subreddit, label, user_emb, sub_emb = batch
            user = user.to(device)
            subreddit = subreddit.to(device)
            label = label.to(device)
            user_emb = user_emb.to(device)
            sub_emb = sub_emb.to(device)

            pred = model(user, subreddit, user_emb, sub_emb)
            loss = criterion(pred, label)

            total_loss += loss.item()
            num_batches += 1

    return total_loss / num_batches

## 6. Main Training Functions

In [ ]:
def train_model_baseline(model_name, data_dir, num_users, num_subreddits):
    """
    Train NCF baseline model (no text embeddings).
    """
    print(f"\n{'='*70}")
    print(f"Training {model_name} (NCF Baseline - No Text)")
    print(f"{'='*70}")
    print(f"  Users: {num_users}, Subreddits: {num_subreddits}")
    print(f"  Data: {data_dir}")

    # Create checkpoint subdirectory for this model
    model_checkpoint_dir = os.path.join(CHECKPOINT_DIR, model_name)
    os.makedirs(model_checkpoint_dir, exist_ok=True)

    # Load datasets
    print("\nLoading datasets...")
    train_dataset = NCFDatasetBaseline(
        os.path.join(data_dir, 'ncf_data', 'train.tsv'),
        num_subreddits, NUM_NEGATIVES
    )
    dev_dataset = NCFDatasetBaseline(
        os.path.join(data_dir, 'ncf_data', 'dev.tsv'),
        num_subreddits, NUM_NEGATIVES
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=2, pin_memory=True)
    dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=True)

    # Initialize model
    model = NCFBaseline(num_users, num_subreddits, EMBEDDING_DIM).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # Training log
    log_file = os.path.join(OUTPUT_DIR, f'{model_name}_training_log.txt')
    best_val_loss = float('inf')
    training_history = []

    with open(log_file, 'w') as f:
        f.write(f"Training {model_name} (NCF Baseline)\n")
        f.write(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"{'='*60}\n")
        f.write(f"Users: {num_users}, Subreddits: {num_subreddits}\n")
        f.write(f"Epochs: {NUM_EPOCHS}, Batch Size: {BATCH_SIZE}, LR: {LEARNING_RATE}\n")
        f.write(f"{'='*60}\n\n")

    print("\nStarting training...")
    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start = datetime.now()

        # Train
        train_loss = train_epoch_baseline(model, train_loader, optimizer, criterion, device)

        # Validate
        val_loss = validate_baseline(model, dev_loader, criterion, device)

        epoch_time = (datetime.now() - epoch_start).total_seconds()

        # Log to console
        print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {epoch_time:.1f}s")

        # Log to file
        with open(log_file, 'a') as f:
            f.write(f"Epoch {epoch}: Train={train_loss:.6f}, Val={val_loss:.6f}, Time={epoch_time:.1f}s\n")

        training_history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})

        # Save checkpoint
        if epoch % CHECKPOINT_EVERY == 0:
            checkpoint_path = os.path.join(model_checkpoint_dir, f'epoch_{epoch}.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss,
            }, checkpoint_path)
            print(f"  → Checkpoint saved: {checkpoint_path}")

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(OUTPUT_DIR, f'{model_name}_best.pt')
            torch.save(model.state_dict(), best_path)
            print(f"  → New best model saved (val_loss: {val_loss:.4f})")

    # Save final model
    final_path = os.path.join(OUTPUT_DIR, f'{model_name}_final.pt')
    torch.save(model.state_dict(), final_path)

    print(f"\n✓ Training complete! Best Val Loss: {best_val_loss:.4f}")

    return best_val_loss, training_history

In [ ]:
def train_model_with_text(model_name, data_dir, embedding_dir, num_users, num_subreddits, use_gating = False):
    """
    Train NCF model with text embeddings.
    """
    print(f"\n{'='*70}")
    print(f"Training {model_name} (NCF + Text Embeddings)")
    print(f"{'='*70}")
    print(f"  Users: {num_users}, Subreddits: {num_subreddits}")
    print(f"  Data: {data_dir}")
    print(f"  Embeddings: {embedding_dir}")

    # Create checkpoint subdirectory for this model
    model_checkpoint_dir = os.path.join(CHECKPOINT_DIR, model_name)
    os.makedirs(model_checkpoint_dir, exist_ok=True)

    # Load embeddings
    print("\nLoading embeddings...")
    user_emb_train = np.load(os.path.join(embedding_dir, 'train', 'user_embeddings_128.npy'))
    sub_emb_train = np.load(os.path.join(embedding_dir, 'train', 'subreddit_embeddings_128.npy'))
    user_emb_dev = np.load(os.path.join(embedding_dir, 'dev', 'user_embeddings_128.npy'))
    sub_emb_dev = np.load(os.path.join(embedding_dir, 'dev', 'subreddit_embeddings_128.npy'))

    print(f"  Train - Users: {user_emb_train.shape}, Subreddits: {sub_emb_train.shape}")
    print(f"  Dev   - Users: {user_emb_dev.shape}, Subreddits: {sub_emb_dev.shape}")

    # Load datasets
    print("\nLoading datasets...")
    # Load subreddit mapping for the ID mapping
    with open(os.path.join(data_dir, 'subreddit_mapping.json'), 'r') as f:
        subreddit_mapping = json.load(f)

    # Load datasets with mapping files
    train_dataset = NCFDatasetWithText(
        os.path.join(data_dir, 'ncf_data', 'train.tsv'),
        user_emb_train, sub_emb_train, num_subreddits, NUM_NEGATIVES,
        user_ids_file=os.path.join(embedding_dir, 'train', 'user_ids.json'),
        subreddit_names_file=os.path.join(embedding_dir, 'train', 'subreddit_names.json'),
        subreddit_mapping=subreddit_mapping
    )
    dev_dataset = NCFDatasetWithText(
        os.path.join(data_dir, 'ncf_data', 'dev.tsv'),
        user_emb_dev, sub_emb_dev, num_subreddits, NUM_NEGATIVES,
        user_ids_file=os.path.join(embedding_dir, 'dev', 'user_ids.json'),
        subreddit_names_file=os.path.join(embedding_dir, 'dev', 'subreddit_names.json'),
        subreddit_mapping=subreddit_mapping
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=2, pin_memory=True)
    dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=True)

    # Initialize model
    if use_gating:
      # Gated Fusion Model
        print("--- Using GATED FUSION Architecture ---")
        model = NCFWithTextGated(
            num_users, num_subreddits,
            EMBEDDING_DIM, TEXT_EMB_DIM, TEXT_PROJ_DIM
        ).to(device)
    else:
      # Standard Concat Model
        print("--- Using STANDARD CONCAT Architecture ---")
        model = NCFWithTextEmbeddings(
            num_users, num_subreddits,
            EMBEDDING_DIM, TEXT_EMB_DIM, TEXT_PROJ_DIM
        ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # Training log
    log_file = os.path.join(OUTPUT_DIR, f'{model_name}_training_log.txt')
    best_val_loss = float('inf')
    training_history = []

    with open(log_file, 'w') as f:
        f.write(f"Training {model_name} (NCF + Text Embeddings)\n")
        f.write(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"{'='*60}\n")
        f.write(f"Users: {num_users}, Subreddits: {num_subreddits}\n")
        f.write(f"Text Embedding Dim: {TEXT_EMB_DIM}, Projected Dim: {TEXT_PROJ_DIM}\n")
        f.write(f"Epochs: {NUM_EPOCHS}, Batch Size: {BATCH_SIZE}, LR: {LEARNING_RATE}\n")
        f.write(f"{'='*60}\n\n")

    print("\nStarting training...")
    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start = datetime.now()

        # Train
        train_loss = train_epoch_with_text(model, train_loader, optimizer, criterion, device)

        # Validate
        val_loss = validate_with_text(model, dev_loader, criterion, device)

        epoch_time = (datetime.now() - epoch_start).total_seconds()

        # Log to console
        print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {epoch_time:.1f}s")

        # Log to file
        with open(log_file, 'a') as f:
            f.write(f"Epoch {epoch}: Train={train_loss:.6f}, Val={val_loss:.6f}, Time={epoch_time:.1f}s\n")

        training_history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})

        # Save checkpoint
        if epoch % CHECKPOINT_EVERY == 0:
            checkpoint_path = os.path.join(model_checkpoint_dir, f'epoch_{epoch}.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss,
            }, checkpoint_path)
            print(f"  → Checkpoint saved: {checkpoint_path}")

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(OUTPUT_DIR, f'{model_name}_best.pt')
            torch.save(model.state_dict(), best_path)
            print(f"  → New best model saved (val_loss: {val_loss:.4f})")

    # Save final model
    final_path = os.path.join(OUTPUT_DIR, f'{model_name}_final.pt')
    torch.save(model.state_dict(), final_path)

    print(f"\n✓ Training complete! Best Val Loss: {best_val_loss:.4f}")

    return best_val_loss, training_history

## 7. Load Mappings

In [ ]:
# Load user and subreddit mappings for each dataset
print("Loading mappings...\n")

# v3.5 tfidf mappings
with open(os.path.join(DATA_DIRS['v3_5tfidf'], 'user_mapping.json'), 'r') as f:
    v3_5_user_mapping = json.load(f)
with open(os.path.join(DATA_DIRS['v3_5tfidf'], 'subreddit_mapping.json'), 'r') as f:
    v3_5_sub_mapping = json.load(f)

# v4 mappings
with open(os.path.join(DATA_DIRS['v4'], 'user_mapping.json'), 'r') as f:
    v4_user_mapping = json.load(f)
with open(os.path.join(DATA_DIRS['v4'], 'subreddit_mapping.json'), 'r') as f:
    v4_sub_mapping = json.load(f)

# v4 chi-sq
with open(os.path.join(DATA_DIRS['v4_chi2'], 'user_mapping.json'), 'r') as f:
    v4_chi2_users = json.load(f)['num_users']
with open(os.path.join(DATA_DIRS['v4_chi2'], 'subreddit_mapping.json'), 'r') as f:
    v4_chi2_subs = json.load(f)['num_subreddits']

print(f"v3.5 (tfidf): {v3_5_user_mapping['num_users']} users, {v3_5_sub_mapping['num_subreddits']} subreddits")
print(f"v4:           {v4_user_mapping['num_users']} users, {v4_sub_mapping['num_subreddits']} subreddits")

## 8. Train All Models

In [ ]:
# Store results for comparison
results = {}

In [ ]:
# ============================================================
# MODEL A: v3.5 data + 128-dim text embeddings
# ============================================================

best_loss_a, history_a = train_model_with_text(
    model_name='model_A_v35_text',
    data_dir=DATA_DIRS['v3_5tfidf'],
    embedding_dir=EMBEDDING_DIRS['v3_5tfidf_filtered'],
    num_users=v3_5_user_mapping['num_users'],
    num_subreddits=v3_5_sub_mapping['num_subreddits']
)

results['Model A (v3.5 + Text)'] = best_loss_a

In [ ]:
# ============================================================
# MODEL B: v4 data + NCF only (BASELINE)
# ============================================================

best_loss_b, history_b = train_model_baseline(
    model_name='model_B_v4_baseline',
    data_dir=DATA_DIRS['v4'],
    num_users=v4_user_mapping['num_users'],
    num_subreddits=v4_sub_mapping['num_subreddits']
)

results['Model B (v4 Baseline)'] = best_loss_b

In [ ]:
# ============================================================
# MODEL C: v4 data + 128-dim text embeddings
# ============================================================

best_loss_c, history_c = train_model_with_text(
    model_name='model_C_v4_text',
    data_dir=DATA_DIRS['v4'],
    embedding_dir=EMBEDDING_DIRS['v4_filtered'],
    num_users=v4_user_mapping['num_users'],
    num_subreddits=v4_sub_mapping['num_subreddits']
)

results['Model C (v4 + Text)'] = best_loss_c

In [ ]:
# Model D: TF-IDF + Gated Fusion
# (Uses v3.5tfidf data, but Gated model)

best_loss_d, _ = train_model_with_text(
    model_name='model_D_tfidf_gated',
    data_dir=DATA_DIRS['v3_5tfidf'],
    embedding_dir=EMBEDDING_DIRS['v3_5tfidf_filtered'],
    num_users=v3_5_user_mapping['num_users'],
    num_subreddits=v3_5_sub_mapping['num_subreddits'],
    use_gating=True # Enable gating
)
results['Model D (TF-IDF + Gated)'] = best_loss_d

In [ ]:
# Model E: v4 TF-IDF + Gated Fusion

best_loss_g, _ = train_model_with_text(
    model_name='model_E_v4_tfidf_gated',
    data_dir=DATA_DIRS['v4'],
    embedding_dir=EMBEDDING_DIRS['v4_filtered'],
    num_users=v4_user_mapping['num_users'],
    num_subreddits=v4_sub_mapping['num_subreddits'],
    use_gating=True
)
results['Model E (v4 TF-IDF + Gated)'] = best_loss_g

In [ ]:
# Model F: Chi-Square + Non-Gated (Concat)
# (Uses new v4_chi2 data, but standard Concat model)

with open(os.path.join(DATA_DIRS['v4_chi2'], 'user_mapping.json'), 'r') as f:
    v4_chi2_users = json.load(f)['num_users']
with open(os.path.join(DATA_DIRS['v4_chi2'], 'subreddit_mapping.json'), 'r') as f:
    v4_chi2_subs = json.load(f)['num_subreddits']

best_loss_e, _ = train_model_with_text(
    model_name='model_F_chi2_concat',
    data_dir=DATA_DIRS['v4_chi2'],
    embedding_dir=EMBEDDING_DIRS['v4_chi2_filtered'],
    num_users=v4_chi2_users,
    num_subreddits=v4_chi2_subs,
    use_gating=False # Standard Concat
)
results['Model F (Chi2 + Concat)'] = best_loss_e

In [ ]:
# 3. Model G: Chi-Square + Gated Fusion
# (Uses new v4_chi2 data AND Gated model)
with open(os.path.join(DATA_DIRS['v4_chi2'], 'user_mapping.json'), 'r') as f:
    v4_chi2_users = json.load(f)['num_users']
with open(os.path.join(DATA_DIRS['v4_chi2'], 'subreddit_mapping.json'), 'r') as f:
    v4_chi2_subs = json.load(f)['num_subreddits']

best_loss_f, _ = train_model_with_text(
    model_name='model_G_chi2_gated',
    data_dir=DATA_DIRS['v4_chi2'],
    embedding_dir=EMBEDDING_DIRS['v4_chi2_filtered'],
    num_users=v4_chi2_users,
    num_subreddits=v4_chi2_subs,
    use_gating=True  # Enable Gating
)
results['Model G (Chi2 + Gated)'] = best_loss_f

## 9. Results Summary

In [ ]:
print("\n" + "="*70)
print("TRAINING COMPLETE - RESULTS SUMMARY")
print("="*70)

print("\nBest Validation Loss by Model:")
print("-"*40)
for model_name, val_loss in results.items():
    print(f"  {model_name}: {val_loss:.4f}")

# Compare text vs baseline for v4
if 'Model B (v4 Baseline)' in results and 'Model C (v4 + Text)' in results:
    baseline = results['Model B (v4 Baseline)']
    with_text = results['Model C (v4 + Text)']
    improvement = (baseline - with_text) / baseline * 100
    print(f"\nv4 Text Embeddings Improvement: {improvement:.2f}%")
    if improvement > 0:
        print("  → Text embeddings improved performance!")
    else:
        print("  → Baseline performed better")

print(f"\nModels saved to: {OUTPUT_DIR}")
print(f"Checkpoints saved to: {CHECKPOINT_DIR}")

In [ ]:
# List all saved files
print("\nSaved Files:")
print("="*50)

print("\nModels:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith('.pt') or f.endswith('.txt'):
        filepath = os.path.join(OUTPUT_DIR, f)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"  {f} ({size_mb:.2f} MB)")

print("\nCheckpoints:")
for model_dir in sorted(os.listdir(CHECKPOINT_DIR)):
    model_path = os.path.join(CHECKPOINT_DIR, model_dir)
    if os.path.isdir(model_path):
        checkpoints = sorted(os.listdir(model_path))
        print(f"  {model_dir}/: {checkpoints}")

In [ ]:
# Plot training curves (optional)
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

histories = [
        # Original models
        ('Model A (v3.5 + Text)', history_a),
        ('Model B (v4 Baseline)', history_b),
        ('Model C (v4 + Text)', history_c),

        # NEW models
        ('Model D (v3.5 + Gated)', history_d),
        ('Model E (v4 TF-IDF + Gated)', history_e),
        ('Model F (v4 Chi2 + Concat)', history_f),
        ('Model G (v4 Chi2 + Gated)', history_g),
    ]

    for ax, (name, history) in zip(axes, histories):
        epochs = [h['epoch'] for h in history]
        train_losses = [h['train_loss'] for h in history]
        val_losses = [h['val_loss'] for h in history]

        ax.plot(epochs, train_losses, 'b-', label='Train Loss')
        ax.plot(epochs, val_losses, 'r-', label='Val Loss')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title(name)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()

    # Save figure
    fig_path = os.path.join(OUTPUT_DIR, 'training_curves.png')
    plt.savefig(fig_path, dpi=150)
    print(f"Training curves saved to: {fig_path}")

    plt.show()

except ImportError:
    print("matplotlib not available - skipping plot")